# Semana 5: Práctica. Panel de comparables mineras

**Curso:** Tópicos de Finanzas Avanzadas (ECON-421, UPAO 2026-20)

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/JonathanRosasV/topicos-finanzas-upao/blob/main/05_multiplos_comparables/clase05_practica.ipynb)

Hoy construimos un panel real de comparables para valorar SCCO por múltiplos y triangular contra nuestro DCF de la semana 4. Este es exactamente el "método de contraste" que pide el TR1.

**Requisito previo:** `git pull` en tu fork para tener `utils/finanzas.py` actualizado.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

import numpy as np
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt

from utils.finanzas import valor_por_multiplo_ev, per_justificado

OBJETIVO = "SCCO"
COMPARABLES = ["SCCO", "FCX", "BHP", "RIO", "TECK"]   # mineras grandes y liquidas

## 1. Descargar el panel

Para cada empresa: capitalización, deuda total, caja, EBITDA, utilidad neta y valor libro. Dos cuidados de un panel internacional: los nombres de las filas cambian entre empresas (por eso `fila()` acepta varios candidatos) y no todas reportan en la moneda en que cotizan (TECK cotiza en dólares en Nueva York pero publica sus estados en dólares canadienses), así que convertimos los estados a la moneda del precio antes de calcular cualquier múltiplo. Envolvemos la descarga en una función defensiva: si a una empresa le falta un dato, la reportamos y seguimos (en paneles reales siempre falta algo).

In [ ]:
def fila(df, *nombres):
    for n in nombres:
        if n in df.index:
            return float(df.loc[n].iloc[0])
    raise KeyError(nombres)

def tipo_cambio(moneda_estados, moneda_precio):
    """Factor para llevar los estados financieros a la moneda en que cotiza la accion."""
    if moneda_estados == moneda_precio:
        return 1.0
    return float(yf.Ticker(f"{moneda_estados}{moneda_precio}=X").fast_info["lastPrice"])

datos = {}
for tic in COMPARABLES:
    try:
        tk = yf.Ticker(tic)
        est, bal, info = tk.income_stmt, tk.balance_sheet, tk.info
        m_est, m_px = info.get("financialCurrency", "USD"), info.get("currency", "USD")
        fx = tipo_cambio(m_est, m_px)
        print(f"{tic}: cierre fiscal {est.columns[0].date()} | estados en {m_est}, precio en {m_px} | factor {fx:.4f}")
        datos[tic] = {
            "cap (MM)": tk.fast_info["marketCap"] / 1e6,
            "deuda (MM)": fila(bal, "Total Debt") * fx / 1e6,
            "caja (MM)": fila(bal, "Cash And Cash Equivalents",
                              "Cash Cash Equivalents And Short Term Investments") * fx / 1e6,
            "EBITDA (MM)": fila(est, "EBITDA", "Normalized EBITDA") * fx / 1e6,
            "NI (MM)": fila(est, "Net Income", "Net Income Common Stockholders") * fx / 1e6,
            "libro (MM)": fila(bal, "Stockholders Equity", "Common Stock Equity") * fx / 1e6,
        }
    except Exception as e:
        print(f"{tic}: dato faltante ({e}); se excluye del panel")

panel = pd.DataFrame(datos).T
panel.round(0)

## 2. Calcular los múltiplos

Todos con la métrica del último año fiscal reportado y la capitalización de hoy, todos con las mismas definiciones: consistencia ante todo. Ojo: esto no es un trailing estricto de 12 meses (para eso habría que sumar los últimos cuatro trimestres, y BHP y RIO reportan por semestres). Mira los cierres fiscales que imprimió la celda anterior: BHP cierra en junio y el resto en diciembre. En un informe profesional ese desfase se declara.

In [ ]:
panel["EV (MM)"] = panel["cap (MM)"] + panel["deuda (MM)"] - panel["caja (MM)"]
panel["EV/EBITDA"] = panel["EV (MM)"] / panel["EBITDA (MM)"]
panel["P/E"] = panel["cap (MM)"] / panel["NI (MM)"]
panel["P/B"] = panel["cap (MM)"] / panel["libro (MM)"]

multiplos = panel[["EV/EBITDA", "P/E", "P/B"]].round(1)
multiplos.loc["MEDIANA"] = multiplos.median()
multiplos

**Preguntas de discusión:** ¿hay algún outlier evidente? ¿La mediana cambia mucho si lo excluyes? ¿Por qué preferimos la mediana al promedio? (Prueba: `panel[["EV/EBITDA"]].mean()` contra la mediana.)

## 3. Valorar SCCO por la mediana del grupo

Regla de oro: la mediana EV/EBITDA se aplica al EBITDA de SCCO, da un EV, y de ahí cruzamos el puente al equity. Excluimos a SCCO de su propia mediana (no puede ser su propio comparable).

In [ ]:
med = panel.drop(OBJETIVO)[["EV/EBITDA", "P/E"]].median()
print(f"Medianas sin {OBJETIVO}: EV/EBITDA = {med['EV/EBITDA']:.1f}x | P/E = {med['P/E']:.1f}x")

ebitda_obj = panel.loc[OBJETIVO, "EBITDA (MM)"]
deuda_neta = panel.loc[OBJETIVO, "deuda (MM)"] - panel.loc[OBJETIVO, "caja (MM)"]
acciones = yf.Ticker(OBJETIVO).fast_info["shares"] / 1e6
px = yf.Ticker(OBJETIVO).fast_info["lastPrice"]

v_ev = valor_por_multiplo_ev(med["EV/EBITDA"], ebitda_obj, deuda_neta, acciones)
v_pe = med["P/E"] * panel.loc[OBJETIVO, "NI (MM)"] / acciones

print(f"Por EV/EBITDA: {v_ev:,.2f} | Por P/E: {v_pe:,.2f} | Precio de mercado: {px:,.2f}")

## 4. La triangulación de la unidad

Juntamos en un solo gráfico todo lo que sabemos de SCCO: los dos DCF de la semana 4 (copia aquí tus resultados) y los dos múltiplos de hoy, contra el precio de mercado.

In [ ]:
# Copia aqui tus valores por accion del DCF de la practica de la semana 4:
V_DCF_GORDON = None      # ejemplo: 61.40
V_DCF_MULTIPLO = None    # ejemplo: 74.20

metodos = {"Comparables EV/EBITDA": v_ev, "Comparables P/E": v_pe}
if V_DCF_GORDON: metodos["DCF (Gordon)"] = V_DCF_GORDON
if V_DCF_MULTIPLO: metodos["DCF (multiplo salida)"] = V_DCF_MULTIPLO

fig, ax = plt.subplots(figsize=(8, 4))
nombres, valores = list(metodos), list(metodos.values())
ax.barh(nombres, valores, color="#000080", alpha=0.85)
ax.axvline(px, color="#E36C0A", lw=2, label=f"Precio de mercado ({px:,.0f})")
for i, v in enumerate(valores):
    ax.text(v, i, f" {v:,.1f}", va="center")
ax.set_title(f"{OBJETIVO}: triangulacion de metodos de valoracion")
ax.legend(); plt.tight_layout(); plt.show()

**Lectura profesional:** ¿el precio cae dentro del rango de tus métodos o fuera? ¿Los métodos intrínsecos y los relativos apuntan en la misma dirección? Si divergen mucho, ¿qué hipótesis lo explicaría (sector caro o barato, supuestos del DCF, comparables imperfectos)?

Una pista para SCCO: mira su fila en la tabla de múltiplos contra las del grupo. Cuando una empresa cotiza de forma persistente con prima sobre sus comparables, la pregunta es si la prima tiene fundamento (en su caso se suelen citar la vida de sus reservas, sus costos bajos y el poco capital flotante, porque Grupo México controla cerca del 89 por ciento) o si es sobrevaloración. La mediana del grupo no responde esa pregunta: solo te dice cuánto valdría SCCO si fuera una minera promedio.

## 5. Cierre

Desde hoy quedan en la librería: `per_justificado()` y `valor_por_multiplo_ev()`. La unidad 1 está completa: intrínseca más relativa más triangulación.

**Tarea** (`clase05_tarea.ipynb`): el panel de comparables de tu empresa; es la pieza que le falta a tu TR1. Entrega hasta el lunes, vía commit.

**TR1: miércoles 07/10 antes de la sesión.** PDF a Canvas más notebook en el fork. La consigna está en `tr1_informe_valoracion/consigna.md`.

**Próxima semana:** empieza la unidad 2, riesgo y rendimiento: por qué diversificar funciona, medido con datos.